# From a trade book to netting-set exposure and XVA

Today's portfolio PV adds trade values. Future credit exposure first adds trade
values **within a legal netting set on each path**, then takes the positive or
negative part. Offsetting trades can therefore reduce exposure. This notebook
compares the same two swaps held together with those swaps held in separate
netting sets. [Setup](../README.md) · [Portfolio PV and risk](../sensitivity/portfolio_risk.ipynb).

`XvaConfig` controls the simulation. Each `NettingSet` carries its own `CsaTerms`
for collateral index/currency and credit/funding inputs. These terms are
separate from the contracts and from the model specification.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import quantsupport as qs
from market_data import load_market, read_configuration

market = load_market()
ref = market.quotes.reference_date
config_spec = read_configuration(
    "cva",
    "xva.json",
)
config = qs.XvaConfig.from_dict(config_spec)
config_spec


[xva.json](data/xva.json) configures a calibrated SOFR LGM model, 512 paths, seed
42, and quarterly exposure dates. These modest settings make the example quick
to run; refine paths and dates when assessing numerical precision. The engine
generates the future valuations it needs. It does not consume the DataFrame
produced by `Simulation.paths()` in the earlier notebook.

The receive-fixed and pay-fixed positions below share a maturity but differ in
notional. `credit_spread=0.01` means 100 bp; `recovery=0.40` means 40%. We use
the flat credit/funding inputs supported by `CsaTerms`. A configured credit
curve is another possible input. The Python wrapper does not expose a full
variation-margin agreement with thresholds and margin-call mechanics here.

In [ ]:
def swap(
    identifier,
    notional,
    side,
):
    return qs.Swap(
        identifier=identifier,
        start_date=ref,
        maturity_date=ref + "3Y",
        trade_date=ref,
        notional=notional,
        fixed_rate=0.045,
        currency=qs.Currency.USD,
        market_index=qs.MarketIndex.SOFR,
        side=side,
    )


receive = swap(
    "RECEIVE",
    1_000_000,
    qs.Side.LongReceive,
)
pay = swap(
    "PAY",
    800_000,
    qs.Side.PayShort,
)
csa = qs.CsaTerms(
    collateral_index=qs.MarketIndex.SOFR,
    collateral_currency=qs.Currency.USD,
    credit_spread=0.01,
    recovery=0.40,
    funding_spread=0.005,
)


def run_exposure(netting_sets):
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=market.curves,
        csa_policy=market.csa_policy,
        volatility_surfaces=market.volatility_surfaces,
    )
    context.initialize()

    with context:
        return context.run_xva(
            config,
            netting_sets=netting_sets,
        )


netted = run_exposure(
    [
        qs.NettingSet(
            "ONE-CLIENT",
            [receive, pay],
            csa,
        )
    ]
)
separate = run_exposure(
    [
        qs.NettingSet(
            "CLIENT-A",
            [receive],
            csa,
        ),
        qs.NettingSet(
            "CLIENT-B",
            [pay],
            csa,
        ),
    ]
)
netted.xva_values


## Retrieve the three result views

`xva_values` is a table keyed by netting set and measure. `exposures` is a list
of profiles; select them by `netting_set`, rather than assuming list order.
`sensitivities` is the XVA parameter-risk table, separate from today's trade
PV sensitivities.

For this API, EPE is the expected positive value at each date, ENE is the
expected negative value **with a negative sign**, and EE is the signed mean.
Consequently `EE = EPE + ENE`. EPE here is a date profile, not a single
time-averaged scalar. The Python result exposes these summaries, not the full
pathwise NPV cube or a PFE-percentile method.

In [ ]:
netted_profiles = {
    profile.netting_set: profile.to_dataframe().set_index("date")
    for profile in netted.exposures
}
separate_profiles = {
    profile.netting_set: profile.to_dataframe().set_index("date")
    for profile in separate.exposures
}
profile = netted_profiles["ONE-CLIENT"]
separate_total = separate_profiles["CLIENT-A"] + separate_profiles["CLIENT-B"]
assert np.allclose(
    profile["ee"],
    profile["epe"] + profile["ene"],
)
assert (profile["epe"] >= -1e-8).all() and (profile["ene"] <= 1e-8).all()
assert (profile["epe"] <= separate_total["epe"] + 1e-6).all()
profile.head(8)


In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)
dates = pd.to_datetime(profile.index)
for measure in ["epe", "ene", "ee"]:
    axes[0].plot(
        dates,
        profile[measure],
        label=measure.upper(),
    )
axes[0].set(
    title="One client: signed exposure profiles",
    xlabel="Date",
    ylabel="USD exposure",
)
axes[0].axhline(
    0,
    color="black",
    linewidth=0.7,
)
axes[0].legend()
axes[1].plot(
    dates,
    profile["epe"],
    label="One netting set",
)
axes[1].plot(
    dates,
    separate_total["epe"],
    label="Two separate netting sets",
)
axes[1].set(
    title="Effect of netting on EPE",
    xlabel="Date",
    ylabel="USD expected positive exposure",
)
axes[1].legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


The swaps offset before the positive-part operation when they are in one
netting set. In separate sets, positive exposure to one client cannot be
reduced by a liability to another. The example holds market/model and client
parameters fixed so that the grouping is the economic difference.

The XVA table below aggregates each run by measure for comparison. Preserve the
engine's measure labels and sign conventions in downstream reporting, and do
not add every reported number to PV without deciding how your report defines
valuation adjustments. Credit spread, recovery, funding spread, and market
quotes also have different risk units.

In [ ]:
pd.concat(
    {
        "one netting set": netted.xva_values.groupby("measure")["value"].sum(),
        "separate sets": separate.xva_values.groupby("measure")["value"].sum(),
    },
    axis=1,
)


In [ ]:
netted.sensitivities.sort_values(
    "value",
    key=lambda series: series.abs(),
    ascending=False,
).head(12)


Increase path count and refine the exposure grid in the configuration to study
stability. Changing the seed helps reveal sampling variation. This example
isolates one-currency rate exposure; it does not establish the adequacy of a
model, calibration basket, or credit/funding assumption for another portfolio.